# Gráficos Descriptivos del Dataset Pokémon

## Importar librerías necesarias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Crear carpeta de outputs si no existe
os.makedirs('outputs', exist_ok=True)

# Configurar el estilo de los gráficos
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Cargar el dataset
df = pd.read_excel('dataset/Pokemon.xlsx')
print(f"Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")

ModuleNotFoundError: No module named 'matplotlib'

## 1. Distribuciones de Variables Numéricas

In [ ]:
# Obtener columnas numéricas
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

# Crear subplots para histogramas
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Distribución de Variables Numéricas', fontsize=16, fontweight='bold')

for idx, col in enumerate(numeric_cols[:6]):
    row = idx // 3
    col_idx = idx % 3
    axes[row, col_idx].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[row, col_idx].set_title(col, fontweight='bold')
    axes[row, col_idx].set_xlabel('Valor')
    axes[row, col_idx].set_ylabel('Frecuencia')
    axes[row, col_idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Boxplots de Variables Numéricas

In [ ]:
# Crear boxplots para detectar outliers
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Boxplots de Variables Numéricas', fontsize=16, fontweight='bold')

for idx, col in enumerate(numeric_cols[:6]):
    row = idx // 3
    col_idx = idx % 3
    axes[row, col_idx].boxplot(df[col].dropna(), vert=True)
    axes[row, col_idx].set_title(col, fontweight='bold')
    axes[row, col_idx].set_ylabel('Valor')
    axes[row, col_idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/02_boxplots_variables_numericas.png', dpi=300, bbox_inches='tight')
plt.savefig('outputs/01_distribucion_variables_numericas.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Matriz de Correlación con Heatmap

In [ ]:
# Calcular matriz de correlación
correlation_matrix = df.select_dtypes(include=['float64', 'int64']).corr()

# Crear heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, linewidths=1, cbar_kws={'label': 'Correlación'})
plt.title('Matriz de Correlación de Variables Numéricas', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('outputs/03_matriz_correlacion_heatmap.png', dpi=300, bbox_inches='tight')
plt.savefig('outputs/02_boxplots_variables_numericas.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Análisis de Variables Categóricas

In [ ]:
# Obtener columnas categóricas
categorical_cols = df.select_dtypes(include='object').columns

# Crear gráficos de barras para variables categóricas
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Distribución de Variables Categóricas', fontsize=16, fontweight='bold')

for idx, col in enumerate(categorical_cols[:4]):
    row = idx // 2
    col_idx = idx % 2
    
    value_counts = df[col].value_counts().head(10)
    axes[row, col_idx].barh(range(len(value_counts)), value_counts.values, color='coral', edgecolor='black')
    axes[row, col_idx].set_yticks(range(len(value_counts)))
    axes[row, col_idx].set_yticklabels(value_counts.index, fontsize=9)
    axes[row, col_idx].set_xlabel('Frecuencia')
    axes[row, col_idx].set_title(f'{col} (Top 10)', fontweight='bold')
    axes[row, col_idx].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/04_variables_categoricas.png', dpi=300, bbox_inches='tight')
plt.savefig('outputs/03_matriz_correlacion_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Scatterplots de Variables Correlacionadas

In [ ]:
# Encontrar pares de variables con alta correlación
corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_value = correlation_matrix.iloc[i, j]
        if abs(corr_value) > 0.7:
            corr_pairs.append((correlation_matrix.columns[i], correlation_matrix.columns[j], corr_value))

corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)

# Crear scatterplots para las 4 correlaciones más altas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Scatterplots de Variables Altamente Correlacionadas', fontsize=16, fontweight='bold')

for idx, (col1, col2, corr) in enumerate(corr_pairs[:4]):
    row = idx // 2
    col_idx = idx % 2
    
    axes[row, col_idx].scatter(df[col1], df[col2], alpha=0.6, s=30, color='steelblue', edgecolor='black')
    axes[row, col_idx].set_xlabel(col1, fontweight='bold')
    axes[row, col_idx].set_ylabel(col2, fontweight='bold')
    axes[row, col_idx].set_title(f'Correlación: {corr:.3f}', fontweight='bold')
    axes[row, col_idx].grid(alpha=0.3)
    
    # Agregar línea de tendencia
    z = np.polyfit(df[col1].dropna(), df[col2].dropna(), 1)
    p = np.poly1d(z)
    axes[row, col_idx].plot(df[col1].sort_values(), p(df[col1].sort_values()), "r--", alpha=0.8, linewidth=2)

plt.tight_layout()
plt.savefig('outputs/05_scatterplots_correlacionadas.png', dpi=300, bbox_inches='tight')
plt.savefig('outputs/04_variables_categoricas.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Densidad de Distribuciones

In [ ]:
# Crear gráficos de densidad
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Densidad de Variables Numéricas', fontsize=16, fontweight='bold')

for idx, col in enumerate(numeric_cols[:6]):
    row = idx // 3
    col_idx = idx % 3
    
    axes[row, col_idx].hist(df[col].dropna(), bins=30, density=True, alpha=0.6, color='steelblue', edgecolor='black')
    df[col].dropna().plot(kind='kde', ax=axes[row, col_idx], color='red', linewidth=2)
    axes[row, col_idx].set_title(col, fontweight='bold')
    axes[row, col_idx].set_xlabel('Valor')
    axes[row, col_idx].set_ylabel('Densidad')
    axes[row, col_idx].grid(alpha=0.3)
    axes[row, col_idx].legend(['Densidad', 'Histograma'])

plt.tight_layout()
plt.savefig('outputs/06_densidad_variables_numericas.png', dpi=300, bbox_inches='tight')
plt.savefig('outputs/05_scatterplots_correlacionadas.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Pairplot de Variables Seleccionadas

In [ ]:
# Seleccionar las primeras 4 variables numéricas para el pairplot
selected_cols = list(numeric_cols[:4])

# Crear pairplot
pairplot = sns.pairplot(df[selected_cols].dropna(), diag_kind='hist', plot_kws={'alpha': 0.6, 's': 30}, 
                         diag_kws={'bins': 30, 'edgecolor': 'black'})
pairplot.fig.suptitle('Pairplot de Variables Numéricas Seleccionadas', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('outputs/07_pairplot_variables_seleccionadas.png', dpi=300, bbox_inches='tight')
plt.savefig('outputs/06_densidad_variables_numericas.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Resumen Visual

In [ ]:
print("\n" + "="*70)
print("RESUMEN DE GRÁFICOS DESCRIPTIVOS GENERADOS")
print("="*70)
print("\n📊 Gráficos Incluidos:")
print("   1. Histogramas de distribuciones de variables numéricas")
print("   2. Boxplots para detección de outliers")
print("   3. Heatmap de matriz de correlación")
print("   4. Gráficos de barras de variables categóricas")
print("   5. Scatterplots con líneas de tendencia para variables correlacionadas")
print("   6. Gráficos de densidad combinados con histogramas")
print("   7. Pairplot para análisis multivariado")

print("\n📈 Variables Analizadas:")
print(f"   • Numéricas: {', '.join(numeric_cols)}")
print(f"   • Categóricas: {', '.join(categorical_cols)}")

print("\n🔍 Insights Útiles:")
print(f"   • Rango de valores y distribuciones de variables numéricas")
print(f"   • Identificación de outliers mediante boxplots")
print(f"   • Relaciones entre variables numéricas")
print(f"   • Distribuciones de categorías en variables cualitativas")
print(f"   • Tendencias y patrones en datos correlacionados")
print("\n" + "="*70)